In [1]:
# ============================================================
# DSA LAB EXAM PROJECT
# Java Bytecode Subset Simulator - Python Version
# ============================================================
#
# PURPOSE:
# This project simulates a small subset of Java bytecode
# instructions using Python.
#
# Instead of using the real Java Virtual Machine (JVM),
# we create our own small simulator that understands
# selected bytecode instructions.
#
# Required bytecodes:
# ldc, iload, istore, iadd, isub, imul, idiv,
# ifeq, iflt, ifgt
#
# Special instructions:
# read, print
#
# Three required test programs:
# A. Minimum and maximum of 3 integers
# B. Sort n integers in ascending order
# C. Add two 2x2 matrices
#
# This version contains exactly 12 automated tests.
# ============================================================


class BytecodeSimulator:
    """
    Simulator for the required small subset of Java bytecode.

    The simulator contains:
    1. An operand stack
    2. Local variables
    3. A program counter (PC)
    4. Input handling
    5. Output handling
    6. Instruction execution
    """

    # Maximum number of values allowed in the operand stack.
    # This prevents unlimited memory usage.
    MAX_STACK_SIZE = 100

    # Maximum number of local variables available.
    # Example:
    # local variable 0 -> self.locals[0]
    # local variable 1 -> self.locals[1]
    MAX_LOCALS = 100

    def __init__(self, program, inputs=None, show_execution=False, interactive=False):
        """
        Constructor of the BytecodeSimulator.

        It creates and initializes all the important parts
        required for bytecode execution.
        """

        # Stores the executable instructions.
        self.program = []

        # Dictionary used to store labels and their instruction positions.
        #
        # Example:
        # "LOOP:" -> instruction number 10
        self.labels = {}

        # Operand stack.
        #
        # Bytecode instructions use this stack to temporarily
        # store values during calculations.
        self.stack = []

        # Local variable storage.
        #
        # We create 100 local variable positions and initially
        # store 0 in every position.
        self.locals = [0] * self.MAX_LOCALS

        # List of input values supplied to the simulator.
        #
        # Example:
        # inputs = [10, 20, 30]
        self.inputs = list(inputs) if inputs is not None else []

        # Keeps track of which input value should be read next.
        self.input_position = 0

        # Stores values produced by the print instruction.
        self.output = []

        # Program Counter (PC).
        #
        # PC tells us which instruction should be executed next.
        # Execution starts from instruction 0.
        self.pc = 0

        # If True, the simulator displays each instruction
        # and the current stack during execution.
        self.show_execution = show_execution

        # If True, input() can be used when predefined input
        # values are not available.
        self.interactive = interactive

        # Load the program before execution starts.
        self.load_program(program)

    # --------------------------------------------------------
    # LOAD PROGRAM
    # --------------------------------------------------------

    def load_program(self, program):
        """
        Load instructions and collect labels before execution.

        A label is not an executable instruction.
        It simply tells us where a jump should go.
        """

        # Clear any previous program and labels.
        self.program = []
        self.labels = {}

        # Process every line of the supplied program.
        for raw_line in program:

            # Convert the line to a string and remove
            # unnecessary spaces from the beginning and end.
            line = str(raw_line).strip()

            # Ignore empty lines.
            #
            # Also ignore lines that start with '#'
            # because they are comments.
            if not line or line.startswith("#"):
                continue

            # Remove inline comments.
            #
            # Example:
            # "ldc 10 # load ten"
            #
            # becomes:
            # "ldc 10"
            if "#" in line:
                line = line.split("#", 1)[0].strip()

            # The line may become empty after removing
            # the comment, so ignore it.
            if not line:
                continue

            # ------------------------------------------------
            # LABEL PROCESSING
            # ------------------------------------------------
            #
            # A label is written like:
            #
            # LOOP:
            #
            # The colon tells us that this is a label.
            if line.endswith(":"):

                # Remove the final ':'.
                label = line[:-1].strip()

                # A label cannot be empty.
                if not label:
                    raise ValueError("Empty label found.")

                # The same label cannot appear twice.
                if label in self.labels:
                    raise ValueError(f"Duplicate label: {label}")

                # Store the position of the next executable instruction.
                #
                # len(self.program) gives the index where the next
                # instruction will be placed.
                self.labels[label] = len(self.program)

            else:
                # If the line is not a label,
                # it is an executable instruction.
                self.program.append(line)

    # --------------------------------------------------------
    # STACK OPERATIONS
    # --------------------------------------------------------

    def push(self, value):
        """
        Push a value onto the operand stack.
        """

        # Prevent stack overflow.
        if len(self.stack) >= self.MAX_STACK_SIZE:
            raise RuntimeError("Operand stack overflow.")

        # Convert the value into an integer and place it
        # on the top of the stack.
        self.stack.append(int(value))

    def pop(self):
        """
        Remove and return the top value from the operand stack.
        """

        # We cannot pop from an empty stack.
        if not self.stack:
            raise RuntimeError("Operand stack underflow.")

        # pop() removes the last/top value.
        return self.stack.pop()

    def peek(self):
        """
        Return the top value without removing it.
        """

        # There must be at least one value in the stack.
        if not self.stack:
            raise RuntimeError("Operand stack is empty.")

        # [-1] means the last element.
        return self.stack[-1]

    # --------------------------------------------------------
    # READ INSTRUCTION
    # --------------------------------------------------------

    def read_integer(self):
        """
        Implements the special 'read' instruction.

        The instruction obtains an integer and pushes it
        onto the operand stack.
        """

        # First check whether predefined input values exist.
        if self.input_position < len(self.inputs):

            # Get the next input value.
            value = self.inputs[self.input_position]

            # Move to the next input position.
            self.input_position += 1

        # If there are no predefined inputs but interactive
        # mode is enabled, ask the user for an integer.
        elif self.interactive:
            value = int(input("Enter integer: "))

        else:
            # No input is available.
            raise RuntimeError(
                "Input required, but no input value is available."
            )

        # Push the obtained integer onto the operand stack.
        self.push(value)

    # --------------------------------------------------------
    # JAVA INTEGER DIVISION
    # --------------------------------------------------------

    @staticmethod
    def java_integer_division(a, b):
        """
        Performs Java-style integer division.

        Java integer division truncates toward zero.

        Example:
        7 / 2  -> 3
        -7 / 2 -> -3

        Python's // operator behaves differently for
        negative values, so we implement Java-style division
        manually.
        """

        # Division by zero is not allowed.
        if b == 0:
            raise ZeroDivisionError("Division by zero.")

        # Determine the sign of the answer.
        #
        # If a and b have different signs -> negative.
        # Otherwise -> positive.
        sign = -1 if (a < 0) != (b < 0) else 1

        # Divide the absolute values.
        # // gives the integer part.
        return sign * (abs(a) // abs(b))

    # --------------------------------------------------------
    # FIND JUMP TARGET
    # --------------------------------------------------------

    def jump_target(self, label):
        """
        Find the instruction position associated with a label.
        """

        # Make sure the requested label actually exists.
        if label not in self.labels:
            raise RuntimeError(f"Unknown label: {label}")

        # Return the instruction number stored for the label.
        return self.labels[label]

    # --------------------------------------------------------
    # EXECUTE ONE INSTRUCTION
    # --------------------------------------------------------

    def execute_instruction(self, instruction):
        """
        Execute exactly one bytecode instruction.

        The instruction is first divided into:
        opcode + operands.

        Example:

        "ldc 10"

        opcode  = ldc
        operand = 10
        """

        # Split the instruction into individual words.
        parts = instruction.split()

        # The first word is always the opcode.
        #
        # lower() makes the instruction case-insensitive.
        opcode = parts[0].lower()

        # ====================================================
        # ldc
        # ====================================================
        #
        # Load a constant onto the operand stack.
        #
        # Example:
        # ldc 10
        #
        # Stack:
        # [] -> [10]
        if opcode == "ldc":

            # ldc must have exactly one operand.
            if len(parts) != 2:
                raise ValueError("ldc requires one integer.")

            # Convert the operand into an integer
            # and push it onto the stack.
            self.push(int(parts[1]))

            # None means:
            # continue with the next instruction.
            return None

        # ====================================================
        # iload
        # ====================================================
        #
        # Load a value from a local variable onto the stack.
        #
        # Example:
        #
        # locals[2] = 50
        # iload 2
        #
        # Stack:
        # [] -> [50]
        if opcode == "iload":

            if len(parts) != 2:
                raise ValueError("iload requires one local index.")

            # Convert the local variable number to an integer.
            index = int(parts[1])

            # Check whether the index is valid.
            if not 0 <= index < self.MAX_LOCALS:
                raise RuntimeError("Invalid local variable index.")

            # Load the value from locals[index]
            # and push it onto the operand stack.
            self.push(self.locals[index])

            return None

        # ====================================================
        # istore
        # ====================================================
        #
        # Remove the top stack value and store it
        # inside a local variable.
        #
        # Example:
        #
        # Stack = [25]
        # istore 0
        #
        # Result:
        # locals[0] = 25
        # Stack = []
        if opcode == "istore":

            if len(parts) != 2:
                raise ValueError("istore requires one local index.")

            index = int(parts[1])

            # Check whether the local variable index is valid.
            if not 0 <= index < self.MAX_LOCALS:
                raise RuntimeError("Invalid local variable index.")

            # pop() removes the top value from the stack.
            self.locals[index] = self.pop()

            return None

        # ====================================================
        # ARITHMETIC INSTRUCTIONS
        # ====================================================
        #
        # Supported:
        # iadd -> addition
        # isub -> subtraction
        # imul -> multiplication
        # idiv -> division
        if opcode in ("iadd", "isub", "imul", "idiv"):

            # IMPORTANT:
            #
            # The second operand is popped first.
            #
            # Example:
            # Stack = [10, 20]
            #
            # b = 20
            # a = 10
            #
            # Therefore:
            # a - b = 10 - 20
            b = self.pop()
            a = self.pop()

            # ----------------------------
            # ADDITION
            # ----------------------------
            if opcode == "iadd":
                self.push(a + b)

            # ----------------------------
            # SUBTRACTION
            # ----------------------------
            elif opcode == "isub":
                self.push(a - b)

            # ----------------------------
            # MULTIPLICATION
            # ----------------------------
            elif opcode == "imul":
                self.push(a * b)

            # ----------------------------
            # DIVISION
            # ----------------------------
            else:
                self.push(self.java_integer_division(a, b))

            return None

        # ====================================================
        # CONDITIONAL BRANCH INSTRUCTIONS
        # ====================================================
        #
        # ifeq -> jump if value == 0
        # iflt -> jump if value < 0
        # ifgt -> jump if value > 0
        if opcode in ("ifeq", "iflt", "ifgt"):

            # Each branch instruction needs a label.
            if len(parts) != 2:
                raise ValueError(
                    f"{opcode} requires one label."
                )

            # Remove the value from the stack.
            value = self.pop()

            # Check the condition.
            condition = (
                (opcode == "ifeq" and value == 0)
                or (opcode == "iflt" and value < 0)
                or (opcode == "ifgt" and value > 0)
            )

            # If the condition is true,
            # return the instruction number of the label.
            if condition:
                return self.jump_target(parts[1])

            # If the condition is false,
            # continue normally with the next instruction.
            return None

        # ====================================================
        # read
        # ====================================================
        #
        # Read an integer and push it onto the stack.
        if opcode == "read":
            self.read_integer()
            return None

        # ====================================================
        # print
        # ====================================================
        #
        # Print the top value of the stack.
        #
        # Notice that peek() is used instead of pop().
        # Therefore, the value remains on the stack.
        if opcode == "print":

            value = self.peek()

            # Save the output so tests can compare it.
            self.output.append(value)

            # Display the value to the user.
            print(value)

            return None

        # ====================================================
        # UNKNOWN INSTRUCTION
        # ====================================================

        # If execution reaches here, the opcode is not supported.
        raise ValueError(
            f"Unknown instruction '{opcode}' at PC={self.pc}"
        )

    # --------------------------------------------------------
    # RUN PROGRAM
    # --------------------------------------------------------

    def run(self, max_steps=100000):
        """
        Execute the loaded program until it finishes.

        The program counter determines which instruction
        is executed next.
        """

        # Always begin execution from instruction 0.
        self.pc = 0

        # Count how many instructions have been executed.
        steps = 0

        # Continue while PC points to a valid instruction.
        while self.pc < len(self.program):

            # Prevent infinite loops from running forever.
            if steps >= max_steps:
                raise RuntimeError(
                    "Maximum execution steps exceeded."
                )

            # Count the current instruction.
            steps += 1

            # Get the instruction at the current PC.
            instruction = self.program[self.pc]

            # ------------------------------------------------
            # OPTIONAL EXECUTION TRACE
            # ------------------------------------------------
            #
            # This is very useful for debugging and viva.
            # It shows:
            # PC + instruction + current stack
            if self.show_execution:
                print(
                    f"PC={self.pc:03d} | "
                    f"{instruction:<18} | "
                    f"Stack={self.stack}"
                )

            # Execute the current instruction.
            new_pc = self.execute_instruction(instruction)

            # ------------------------------------------------
            # UPDATE PROGRAM COUNTER
            # ------------------------------------------------
            #
            # If execute_instruction() returned None,
            # move to the next instruction.
            #
            # If it returned a number, that number is a
            # jump destination.
            if new_pc is None:
                self.pc += 1
            else:
                self.pc = new_pc

        # Return all values produced by print instructions.
        return self.output


# ============================================================
# HELPER FUNCTION FOR RUNNING PROGRAMS
# ============================================================

def run_program(program, inputs, title, show_execution=False):
    """
    Run one test program with fixed inputs.

    The function:
    1. Displays the test title
    2. Creates a simulator
    3. Runs the program
    4. Returns the output
    """

    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

    # Create the simulator object.
    simulator = BytecodeSimulator(
        program,
        inputs=inputs,
        show_execution=show_execution
    )

    # Execute the program.
    result = simulator.run()

    print("-" * 70)
    print("Returned output:", result)

    # Return the output so the test can verify it.
    return result


# ============================================================
# REQUIRED PROGRAM A:
# MINIMUM AND MAXIMUM OF THREE INTEGERS
# ============================================================

TEST_PROGRAM_A = [

    # --------------------------------------------------------
    # STEP 1: Read a, b and c
    # --------------------------------------------------------

    # Read first number.
    # The value goes onto the stack.
    "read",

    # Remove it from stack and store it in local variable 0.
    # local[0] = a
    "istore 0",

    # Read second number.
    "read",

    # local[1] = b
    "istore 1",

    # Read third number.
    "read",

    # local[2] = c
    "istore 2",


    # --------------------------------------------------------
    # STEP 2: Assume a is the minimum
    # --------------------------------------------------------

    # Load a onto the stack.
    "iload 0",

    # Store it as min.
    # local[3] = min
    "istore 3",


    # --------------------------------------------------------
    # STEP 3: Check whether b < min
    # --------------------------------------------------------

    # Push b.
    "iload 1",

    # Push current min.
    "iload 3",

    # Calculate:
    #
    # b - min
    #
    # If result < 0, then b < min.
    "isub",

    # If b - min < 0,
    # jump to A_SET_MIN_B.
    "iflt A_SET_MIN_B",

    # If condition is false,
    # push 0.
    "ldc 0",

    # 0 == 0, therefore jump to the next check.
    "ifeq A_CHECK_MIN_C",


    # --------------------------------------------------------
    # STEP 4: Set min = b
    # --------------------------------------------------------

    "A_SET_MIN_B:",

    # Load b.
    "iload 1",

    # Store b as the new minimum.
    "istore 3",


    # --------------------------------------------------------
    # STEP 5: Check whether c < min
    # --------------------------------------------------------

    "A_CHECK_MIN_C:",

    # Load c.
    "iload 2",

    # Load current min.
    "iload 3",

    # Calculate c - min.
    "isub",

    # If c - min < 0,
    # then c is smaller than min.
    "iflt A_SET_MIN_C",

    # Otherwise use 0 to perform an unconditional jump.
    "ldc 0",

    "ifeq A_SET_MAX_A",


    # --------------------------------------------------------
    # STEP 6: Set min = c
    # --------------------------------------------------------

    "A_SET_MIN_C:",

    # Load c.
    "iload 2",

    # Store c as the minimum.
    "istore 3",


    # --------------------------------------------------------
    # STEP 7: Assume a is the maximum
    # --------------------------------------------------------

    "A_SET_MAX_A:",

    # Load a.
    "iload 0",

    # local[4] = max
    "istore 4",


    # --------------------------------------------------------
    # STEP 8: Check whether b > max
    # --------------------------------------------------------

    # Load b.
    "iload 1",

    # Load current max.
    "iload 4",

    # Calculate b - max.
    "isub",

    # If result > 0,
    # b is greater than max.
    "ifgt A_SET_MAX_B",

    # Otherwise continue.
    "ldc 0",

    "ifeq A_CHECK_MAX_C",


    # --------------------------------------------------------
    # STEP 9: Set max = b
    # --------------------------------------------------------

    "A_SET_MAX_B:",

    # Load b.
    "iload 1",

    # Store b as max.
    "istore 4",


    # --------------------------------------------------------
    # STEP 10: Check whether c > max
    # --------------------------------------------------------

    "A_CHECK_MAX_C:",

    # Load c.
    "iload 2",

    # Load current max.
    "iload 4",

    # Calculate c - max.
    "isub",

    # If result > 0,
    # c is greater than max.
    "ifgt A_SET_MAX_C",

    # Otherwise go to printing.
    "ldc 0",

    "ifeq A_PRINT",


    # --------------------------------------------------------
    # STEP 11: Set max = c
    # --------------------------------------------------------

    "A_SET_MAX_C:",

    # Load c.
    "iload 2",

    # Store c as max.
    "istore 4",


    # --------------------------------------------------------
    # STEP 12: Print minimum and maximum
    # --------------------------------------------------------

    "A_PRINT:",

    # Load minimum.
    "iload 3",

    # Print minimum.
    "print",

    # Load maximum.
    "iload 4",

    # Print maximum.
    "print",
]


# ============================================================
# REQUIRED PROGRAM B:
# BUBBLE SORT
# ============================================================
#
# Local variables:
#
# 0  = n
# 1  = i
# 2  = j
# 3  = temporary
# 10+ = array elements
#
# Example for n = 5:
#
# locals[10] = arr[0]
# locals[11] = arr[1]
# locals[12] = arr[2]
# locals[13] = arr[3]
# locals[14] = arr[4]
#
# Every generated label is unique.
# There is only ONE shared INCREMENT_J label.
# ============================================================

def create_sort_program(n):
    """
    Dynamically create a bytecode program that sorts n integers.

    The sorting algorithm used is Bubble Sort.
    """

    # n must be an integer.
    # n must also be at least 1.
    if not isinstance(n, int) or n < 1:
        raise ValueError(
            "n must be a positive integer."
        )

    # Local variables 10 onwards are used for array elements.
    #
    # We need enough space for all n elements.
    if 10 + n >= BytecodeSimulator.MAX_LOCALS:
        raise ValueError(
            "n is too large for the available local variables."
        )

    # Start with an empty bytecode program.
    program = []


    # --------------------------------------------------------
    # STEP 1: Read n
    # --------------------------------------------------------

    program += [
        "read",
        "istore 0",
    ]


    # --------------------------------------------------------
    # STEP 2: Read all array elements
    # --------------------------------------------------------
    #
    # Each element is stored starting from local variable 10.

    for i in range(n):

        program += [
            "read",

            # Example:
            # i = 0 -> istore 10
            # i = 1 -> istore 11
            # i = 2 -> istore 12
            f"istore {10 + i}",
        ]


    # --------------------------------------------------------
    # STEP 3: i = 0
    # --------------------------------------------------------

    program += [
        "ldc 0",
        "istore 1",

        # Beginning of the outer loop.
        "B_OUTER_LOOP:",
    ]


    # --------------------------------------------------------
    # STEP 4: Outer loop condition
    # --------------------------------------------------------
    #
    # Bubble sort performs n-1 outer passes.
    #
    # Condition:
    #
    # i < n - 1
    #
    # We calculate:
    #
    # i - (n - 1)
    #
    # If result < 0, continue the loop.

    program += [
        "iload 1",

        # Load n - 1.
        f"ldc {n - 1}",

        # Calculate:
        # i - (n - 1)
        "isub",

        # If negative:
        # i < n - 1
        "iflt B_OUTER_BODY",

        # Otherwise the sorting is complete.
        "ldc 0",

        # Unconditional jump to printing.
        "ifeq B_PRINT",
    ]


    # --------------------------------------------------------
    # STEP 5: Outer loop body
    # --------------------------------------------------------

    program.append("B_OUTER_BODY:")


    # --------------------------------------------------------
    # STEP 6: j = 0
    # --------------------------------------------------------

    program += [
        "ldc 0",
        "istore 2",

        # Beginning of inner loop.
        "B_INNER_LOOP:",
    ]


    # --------------------------------------------------------
    # STEP 7: Inner loop condition
    # --------------------------------------------------------
    #
    # Bubble sort compares adjacent elements:
    #
    # arr[j] and arr[j+1]
    #
    # For each outer pass:
    #
    # j < n - i - 1
    #
    # We calculate:
    #
    # j - n + i + 1
    #
    # If result < 0, continue the inner loop.

    program += [

        # Load j.
        "iload 2",

        # Load n.
        "iload 0",

        # j - n
        "isub",

        # Add i.
        "iload 1",
        "iadd",

        # Add 1.
        "ldc 1",
        "iadd",

        # If negative:
        # j < n - i - 1
        "iflt B_INNER_BODY",


        # ----------------------------------------------------
        # Inner loop finished.
        # Increment i.
        # ----------------------------------------------------

        "ldc 1",
        "iload 1",
        "iadd",
        "istore 1",

        # Unconditional jump back to outer loop.
        "ldc 0",
        "ifeq B_OUTER_LOOP",
    ]


    # --------------------------------------------------------
    # STEP 8: Inner loop body
    # --------------------------------------------------------

    program.append("B_INNER_BODY:")


    # --------------------------------------------------------
    # STEP 9: Find which adjacent pair is being compared
    # --------------------------------------------------------
    #
    # Because the bytecode subset does not contain an
    # instruction for dynamically indexing local variables,
    # the Python program generates a comparison block
    # for every possible value of j.
    #
    # Example:
    #
    # j = 0 -> compare arr[0] and arr[1]
    # j = 1 -> compare arr[1] and arr[2]
    # etc.

    for j in range(n - 1):

        program += [

            # Load current j.
            "iload 2",

            # Load constant j.
            f"ldc {j}",

            # Calculate j - constant.
            "isub",

            # If result is zero,
            # current j equals this particular j.
            f"ifeq B_CHECK_{j}",
        ]


    # --------------------------------------------------------
    # SAFETY FALLBACK
    # --------------------------------------------------------
    #
    # If no comparison block matched,
    # continue with incrementing j.

    program += [
        "ldc 0",
        "ifeq B_INCREMENT_J",
    ]


    # --------------------------------------------------------
    # STEP 10: Comparison blocks
    # --------------------------------------------------------

    for j in range(n - 1):

        # Unique label for this pair.
        program.append(f"B_CHECK_{j}:")

        # ----------------------------------------------------
        # Compare arr[j] and arr[j+1]
        # ----------------------------------------------------
        #
        # We calculate:
        #
        # arr[j] - arr[j+1]
        #
        # If the result is positive:
        #
        # arr[j] > arr[j+1]
        #
        # Therefore, the two values must be swapped.

        program += [

            # Load arr[j].
            f"iload {10 + j}",

            # Load arr[j+1].
            f"iload {10 + j + 1}",

            # Calculate:
            # arr[j] - arr[j+1]
            "isub",

            # If positive, perform swap.
            f"ifgt B_SWAP_{j}",

            # If no swap is required,
            # move to the next j.
            "ldc 0",
            "ifeq B_INCREMENT_J",
        ]


        # ----------------------------------------------------
        # STEP 11: Swap arr[j] and arr[j+1]
        # ----------------------------------------------------

        program.append(f"B_SWAP_{j}:")

        program += [

            # temporary = arr[j]
            f"iload {10 + j}",
            "istore 3",


            # arr[j] = arr[j+1]
            f"iload {10 + j + 1}",
            f"istore {10 + j}",


            # arr[j+1] = temporary
            "iload 3",
            f"istore {10 + j + 1}",


            # Continue with the next j.
            "ldc 0",
            "ifeq B_INCREMENT_J",
        ]


    # --------------------------------------------------------
    # STEP 12: Increment j
    # --------------------------------------------------------
    #
    # j = j + 1

    program += [

        "B_INCREMENT_J:",

        "ldc 1",
        "iload 2",
        "iadd",
        "istore 2",

        # Go back to inner loop.
        "ldc 0",
        "ifeq B_INNER_LOOP",


        # ----------------------------------------------------
        # STEP 13: Print sorted array
        # ----------------------------------------------------

        "B_PRINT:",
    ]


    # Print every sorted array element.
    for i in range(n):

        program += [

            # Load arr[i].
            f"iload {10 + i}",

            # Print arr[i].
            "print",
        ]

    # Return the complete generated bytecode program.
    return program


# ============================================================
# REQUIRED PROGRAM C:
# 2x2 MATRIX ADDITION
# ============================================================
#
# Matrix A:
#
# a11  a12
# a21  a22
#
# Matrix B:
#
# b11  b12
# b21  b22
#
# Result:
#
# c11 = a11 + b11
# c12 = a12 + b12
# c21 = a21 + b21
# c22 = a22 + b22
# ============================================================

TEST_PROGRAM_C = [

    # --------------------------------------------------------
    # Read Matrix A
    # --------------------------------------------------------
    #
    # local[0] = a11
    # local[1] = a12
    # local[2] = a21
    # local[3] = a22

    "read",
    "istore 0",

    "read",
    "istore 1",

    "read",
    "istore 2",

    "read",
    "istore 3",


    # --------------------------------------------------------
    # Read Matrix B
    # --------------------------------------------------------
    #
    # local[4] = b11
    # local[5] = b12
    # local[6] = b21
    # local[7] = b22

    "read",
    "istore 4",

    "read",
    "istore 5",

    "read",
    "istore 6",

    "read",
    "istore 7",


    # --------------------------------------------------------
    # Calculate C = A + B
    # --------------------------------------------------------
    #
    # c11 = a11 + b11

    "iload 0",
    "iload 4",
    "iadd",
    "istore 8",


    # c12 = a12 + b12

    "iload 1",
    "iload 5",
    "iadd",
    "istore 9",


    # c21 = a21 + b21

    "iload 2",
    "iload 6",
    "iadd",
    "istore 10",


    # c22 = a22 + b22

    "iload 3",
    "iload 7",
    "iadd",
    "istore 11",


    # --------------------------------------------------------
    # Print result matrix
    # --------------------------------------------------------
    #
    # Print:
    # c11
    # c12
    # c21
    # c22

    "iload 8",
    "print",

    "iload 9",
    "print",

    "iload 10",
    "print",

    "iload 11",
    "print",
]


# ============================================================
# 12 AUTOMATED TESTS
# ============================================================

def run_all_tests():
    """
    Run all 12 automated tests.

    A test is considered successful when the actual output
    matches the expected output.
    """

    # Number of tests that have passed.
    passed = 0

    # Total number of tests.
    total = 12

    print("\n" + "#" * 70)
    print("RUNNING 12 TESTS")
    print("#" * 70)


    # ========================================================
    # TEST 1
    # REQUIRED PROGRAM A - NORMAL CASE
    # ========================================================

    out = run_program(
        TEST_PROGRAM_A,

        # Input:
        # a = 25
        # b = 7
        # c = 18
        [25, 7, 18],

        "TEST 1 - Minimum/Maximum: normal case"
    )

    # Expected:
    # minimum = 7
    # maximum = 25
    assert out == [7, 25]

    print("TEST 1 PASSED")
    passed += 1


    # ========================================================
    # TEST 2
    # PROGRAM A - NEGATIVE NUMBERS
    # ========================================================

    out = run_program(
        TEST_PROGRAM_A,

        # Input contains negative values.
        [-5, -20, 3],

        "TEST 2 - Minimum/Maximum: negative values"
    )

    # Expected:
    # minimum = -20
    # maximum = 3
    assert out == [-20, 3]

    print("TEST 2 PASSED")
    passed += 1


    # ========================================================
    # TEST 3
    # PROGRAM A - EQUAL VALUES
    # ========================================================

    out = run_program(
        TEST_PROGRAM_A,

        # All three values are equal.
        [8, 8, 8],

        "TEST 3 - Minimum/Maximum: equal values"
    )

    # Minimum and maximum are both 8.
    assert out == [8, 8]

    print("TEST 3 PASSED")
    passed += 1


    # ========================================================
    # TEST 4
    # REQUIRED PROGRAM B - NORMAL SORTING
    # ========================================================

    out = run_program(
        create_sort_program(5),

        # First value = n = 5
        #
        # Array:
        # 25, 7, 18, 3, 12
        [5, 25, 7, 18, 3, 12],

        "TEST 4 - Sorting: normal case"
    )

    # Expected sorted array.
    assert out == [3, 7, 12, 18, 25]

    print("TEST 4 PASSED")
    passed += 1


    # ========================================================
    # TEST 5
    # SORTING - ALREADY SORTED
    # ========================================================

    out = run_program(
        create_sort_program(5),

        # Already sorted array.
        [5, 1, 2, 3, 4, 5],

        "TEST 5 - Sorting: already sorted"
    )

    assert out == [1, 2, 3, 4, 5]

    print("TEST 5 PASSED")
    passed += 1


    # ========================================================
    # TEST 6
    # SORTING - REVERSE ORDER
    # ========================================================

    out = run_program(
        create_sort_program(5),

        # Reverse-order array.
        [5, 9, 7, 5, 3, 1],

        "TEST 6 - Sorting: reverse order"
    )

    # Expected ascending order.
    assert out == [1, 3, 5, 7, 9]

    print("TEST 6 PASSED")
    passed += 1


    # ========================================================
    # TEST 7
    # REQUIRED PROGRAM C - MATRIX ADDITION
    # ========================================================

    out = run_program(
        TEST_PROGRAM_C,

        # Matrix A:
        #
        # 1 2
        # 3 4
        #
        # Matrix B:
        #
        # 5 6
        # 7 8
        #
        # Result:
        #
        # 6  8
        # 10 12
        [1, 2, 3, 4, 5, 6, 7, 8],

        "TEST 7 - 2x2 Matrix Addition"
    )

    assert out == [6, 8, 10, 12]

    print("TEST 7 PASSED")
    passed += 1


    # ========================================================
    # TEST 8
    # iadd
    # ========================================================

    out = run_program(

        # Push 10.
        # Push 20.
        # Add them.
        # Print result.
        ["ldc 10", "ldc 20", "iadd", "print"],

        [],

        "TEST 8 - iadd"
    )

    assert out == [30]

    print("TEST 8 PASSED")
    passed += 1


    # ========================================================
    # TEST 9
    # isub
    # ========================================================

    out = run_program(

        # 30 - 12 = 18
        ["ldc 30", "ldc 12", "isub", "print"],

        [],

        "TEST 9 - isub"
    )

    assert out == [18]

    print("TEST 9 PASSED")
    passed += 1


    # ========================================================
    # TEST 10
    # imul
    # ========================================================

    out = run_program(

        # 6 * 7 = 42
        ["ldc 6", "ldc 7", "imul", "print"],

        [],

        "TEST 10 - imul"
    )

    assert out == [42]

    print("TEST 10 PASSED")
    passed += 1


    # ========================================================
    # TEST 11
    # idiv
    # ========================================================

    out = run_program(

        # 20 / 4 = 5
        ["ldc 20", "ldc 4", "idiv", "print"],

        [],

        "TEST 11 - idiv"
    )

    assert out == [5]

    print("TEST 11 PASSED")
    passed += 1


    # ========================================================
    # TEST 12
    # CONDITIONAL BRANCH INSTRUCTIONS
    # ========================================================
    #
    # This test checks:
    #
    # ifeq
    # iflt
    # ifgt
    #
    # All three conditions are tested in one program.

    branch_program = [

        # ----------------------------------------------------
        # Test ifeq
        # ----------------------------------------------------
        #
        # Push 0.
        # Since 0 == 0, jump to T12_EQ.

        "ldc 0",
        "ifeq T12_EQ",

        # This should NOT execute.
        "ldc 999",
        "print",


        # ----------------------------------------------------
        # ifeq target
        # ----------------------------------------------------

        "T12_EQ:",

        # ----------------------------------------------------
        # Test iflt
        # ----------------------------------------------------
        #
        # -1 < 0, therefore jump to T12_LT.

        "ldc -1",
        "iflt T12_LT",

        # This should NOT execute.
        "ldc 999",
        "print",


        # ----------------------------------------------------
        # iflt target
        # ----------------------------------------------------

        "T12_LT:",

        # ----------------------------------------------------
        # Test ifgt
        # ----------------------------------------------------
        #
        # 1 > 0, therefore jump to T12_GT.

        "ldc 1",
        "ifgt T12_GT",

        # This should NOT execute.
        "ldc 999",
        "print",


        # ----------------------------------------------------
        # ifgt target
        # ----------------------------------------------------

        "T12_GT:",

        # If all three branch instructions worked correctly,
        # execution finally reaches here.

        "ldc 12",
        "print",
    ]


    out = run_program(
        branch_program,
        [],
        "TEST 12 - ifeq + iflt + ifgt"
    )

    # Only 12 should be printed.
    #
    # The 999 values must never be printed because
    # all three conditional branches should work correctly.
    assert out == [12]

    print("TEST 12 PASSED")
    passed += 1


    # ========================================================
    # FINAL RESULT
    # ========================================================

    print("\n" + "=" * 70)

    # Display number of successful tests.
    print(
        f"FINAL RESULT: {passed}/{total} TESTS PASSED"
    )

    print("=" * 70)


    # If even one test failed, report failure.
    if passed != total:
        raise AssertionError(
            f"Only {passed}/{total} tests passed."
        )


    # If execution reaches here,
    # every test has passed successfully.
    print("ALL 12 TESTS PASSED SUCCESSFULLY!")

    print("PROJECT STATUS: READY FOR SUBMISSION")


# ============================================================
# START ALL TESTS
# ============================================================
#
# Calling run_all_tests() starts the complete project.
#
# It will:
# 1. Create the simulator
# 2. Run Program A
# 3. Run Bubble Sort
# 4. Run Matrix Addition
# 5. Test arithmetic instructions
# 6. Test conditional instructions
# 7. Verify all 12 tests
# 8. Display the final result
# ============================================================

run_all_tests()


######################################################################
RUNNING 12 TESTS
######################################################################

TEST 1 - Minimum/Maximum: normal case
7
25
----------------------------------------------------------------------
Returned output: [7, 25]
TEST 1 PASSED

TEST 2 - Minimum/Maximum: negative values
-20
3
----------------------------------------------------------------------
Returned output: [-20, 3]
TEST 2 PASSED

TEST 3 - Minimum/Maximum: equal values
8
8
----------------------------------------------------------------------
Returned output: [8, 8]
TEST 3 PASSED

TEST 4 - Sorting: normal case
3
7
12
18
25
----------------------------------------------------------------------
Returned output: [3, 7, 12, 18, 25]
TEST 4 PASSED

TEST 5 - Sorting: already sorted
1
2
3
4
5
----------------------------------------------------------------------
Returned output: [1, 2, 3, 4, 5]
TEST 5 PASSED

TEST 6 - Sorting: reverse order
1
3
5
7
9
---